# 12 — Support Mechanisms: CfD, PPA, FIT and Market Premium compared to naive bidding

In this notebook we compare five identical solar units that differ only in how they monetize
their generation:

1. **CfD** (Contract for Difference) — receives the difference between a fixed strike price and the market price for its contracted generation (two-sided: it pays back when the market price is above the strike).
2. **PPA** (Power Purchase Agreement) — sells its generation at a fixed price to a buyer.
3. **FIT** (Feed-in Tariff) — receives a fixed payment per MWh generated.
4. **Market Premium** — receives the difference between the strike price and the market price, but only when the market price is below the strike (one-sided).
5. **Naive bidding** — no support; bids its marginal cost (~0 €/MWh for solar) into the energy-only market and takes the full price risk.

All units sell their energy on the hourly day-ahead market (**EOM**). The four supported units additionally
bid into a **contract market** (`PayAsBidContractRole`), where the demand side acts as the buyer of the
support contracts (representing the state or a corporate buyer). Contract payments are settled
retrospectively based on the actual generation and the realized EOM price.

The rest of the system is deliberately simple:

- **1 gas power plant** with enough capacity to cover the whole demand at any time (the price-setting unit),
- **1 inelastic demand**.

The simulation runs for the **full year 2019** (hourly resolution). The evaluation at the end of the
notebook compares one week per season: winter, spring, summer and autumn.

In [ ]:
import importlib.util

# Check whether notebook is run in google colab
IN_COLAB = importlib.util.find_spec("google.colab") is not None

if IN_COLAB:
    %pip install assume-framework

## 1. Imports

In [ ]:
import os
from datetime import datetime, timedelta

import matplotlib.pyplot as plt
import pandas as pd
from dateutil import rrule as rr
from dateutil import relativedelta as rd

from assume import World
from assume.common.forecaster import DemandForecaster, PowerplantForecaster
from assume.common.market_objects import MarketConfig, MarketProduct
from assume.markets.clearing_algorithms.contracts import PayAsBidContractRole
from assume.strategies.extended import SupportStrategy

os.makedirs("local_db", exist_ok=True)

db_uri = "sqlite:///local_db/assume_db.db"

## 2. Load input data

We use two data sources from the existing examples:

- **Demand** from `example_01a` (`demand_df.csv`) — an inelastic demand profile for the full year 2019.
- **Solar availability** from `example_03` (`availability_df.csv`) — the solar capacity factor time series for 2019.

Both files are given in 15-minute resolution and are resampled to the hourly resolution of the simulation.

In [ ]:
# paths to the example data (relative to the examples/notebooks folder)
input_01a = "../inputs/example_01a"
input_03 = "../inputs/example_03"

# load demand and availability (15 min resolution) and resample to hourly
demand_df = pd.read_csv(f"{input_01a}/demand_df.csv", index_col=0, parse_dates=True)
availability_df = pd.read_csv(f"{input_03}/availability_df.csv", index_col=0, parse_dates=True)

demand_df = demand_df.resample("1h").mean()
availability_df = availability_df.resample("1h").mean()

solar_availability = availability_df["Solar"]

print(f"Demand: min {demand_df['demand_EOM'].min():.0f} MW, max {demand_df['demand_EOM'].max():.0f} MW")
print(f"Solar availability: min {solar_availability.min():.2f}, max {solar_availability.max():.2f}")

### Quick look at the input data

In [ ]:
fig, ax1 = plt.subplots(figsize=(12, 4))

ax1.plot(demand_df.index, demand_df["demand_EOM"], color="tab:blue", label="Demand (MW)")
ax1.set_ylabel("Demand (MW)")
ax1.set_xlabel("")

ax2 = ax1.twinx()
ax2.plot(solar_availability.index, solar_availability, color="tab:orange", alpha=0.5, label="Solar availability")
ax2.set_ylabel("Solar availability (p.u.)")

fig.legend(loc="upper left", bbox_to_anchor=(0.12, 0.95))
fig.suptitle("Demand and solar availability in 2019")
fig.tight_layout()

## 3. Simulation setup

The simulation covers the full year 2019 with hourly time steps. We use a local sqlite database so no
docker infrastructure is required.

In [ ]:
start = datetime(2019, 1, 1)
end = datetime(2020, 1, 1)
simulation_id = "support_mechanisms"

index = pd.date_range(start=start, end=end, freq="h")

world = World(database_uri=db_uri, export_csv_path="outputs")

world.setup(
    start=start,
    end=end,
    save_frequency_hours=48,
    simulation_id=simulation_id,
    index=index,
)

## 4. Market design

Two markets are used:

### Energy-Only Market (EOM)
A standard hourly day-ahead auction with `pay_as_clear` clearing. All units (solar and gas) sell their
energy here; the demand buys here.

### Contract market ("Support")
A market with the `pay_as_bid_contract` clearing mechanism. Instead of energy delivery, **support
policies** are auctioned here. Units bid for a contract of a specific type (`CFD`, `PPA`, `FIT`,
`market_premium`) with a strike price (`contract_value`). The demand side bids as the buyer of these
contracts. Once a contract is cleared, the market role schedules recurring tasks which settle the
contract retrospectively — based on the actual generation of the selling unit and the realized EOM price.

The contract market opens **monthly** and each contract covers one month, but the cashflow is evaluated
**weekly** (see `evaluation_frequency`).

In [ ]:
# the support strategy is not registered by default and must be added manually
world.bidding_strategies["support"] = SupportStrategy

marketdesign = [
    MarketConfig(
        market_id="EOM",
        opening_hours=rr.rrule(rr.HOURLY, dtstart=start, until=end),
        opening_duration=timedelta(hours=1),
        market_mechanism="pay_as_clear",
        market_products=[MarketProduct(timedelta(hours=1), 1, timedelta(hours=1))],
        product_type="energy",
        volume_unit="MWh",
        price_unit="EUR/MWh",
        maximum_bid_price=3000,
        minimum_bid_price=-500,
        maximum_bid_volume=100000,
    ),
    MarketConfig(
        market_id="Support",
        opening_hours=rr.rrule(rr.MONTHLY, dtstart=start, until=end),
        opening_duration=timedelta(hours=1),
        market_mechanism="pay_as_bid_contract",
        market_products=[MarketProduct(rd.relativedelta(months=1), 1, timedelta(hours=1))],
        product_type="financial_support",
        additional_fields=[
            "sender_id",
            "contract",
            "eligible_lambda",
            "evaluation_frequency",
        ],
        supports_get_unmatched=True,
        param_dict={"allowed_contracts": ["CFD", "PPA", "FIT", "market_premium"]},
        volume_unit="MW",
        price_unit="EUR/MWh",
        maximum_bid_price=3000,
        maximum_bid_volume=100000,
    ),
]

mo_id = "market_operator"
world.add_market_operator(id=mo_id)
for market_config in marketdesign:
    world.add_market(market_operator_id=mo_id, market_config=market_config)

## 5. Units

The units are grouped into three unit operators:

- **gas_operator** — the gas power plant. It is the price-setting unit and must be able to cover the
  whole demand at any time.
- **solar_operator** — the five solar units (1000 MW each). They all use the same availability time
  series, but differ in how they monetize their generation.
- **demand_operator** — the inelastic demand. It additionally acts as the **buyer of the support
  contracts** (representing the state or a corporate buyer paying the support).

> **Note:** the gas plant is placed in its own unit operator because the default eligibility check for
> support contracts (`is_co2emissionless`) requires *all* units of an operator to be emissionless.

The four supported solar units bid into the EOM *and* into the contract market with the `support`
strategy, each for exactly one contract type at a strike price of 80 €/MWh:

| Unit | Contract type | EOM strategy |
|---|---|---|
| `solar_cfd` | `CFD` | naive (marginal cost) |
| `solar_ppa` | `PPA` | naive (marginal cost) |
| `solar_fit` | `FIT` | naive (marginal cost) |
| `solar_mp` | `market_premium` | naive (marginal cost) |
| `solar_naive` | — | naive (marginal cost) |

The contract cashflow is evaluated **weekly** (`evaluation_frequency`), based on the actual generation
of the unit and the realized EOM price.

> **Note on the demand unit:** the `SupportStrategy` uses `min_power` as the contract volume for demand
> units. As `min_power` defaults to 0 (which would result in zero-volume contract bids that can never
> clear), we set it to −4000 MW — the maximum total volume of the four supported solar units. The
> demand bid on the EOM is unaffected, as it is derived from the demand forecast.

In [ ]:
world.add_unit_operator("gas_operator")
world.add_unit_operator("solar_operator")
world.add_unit_operator("demand_operator")

# --- gas plant: covers the whole demand at any time and sets the price ---
gas_forecaster = PowerplantForecaster(
    index,
    availability=1,
    fuel_prices={"natural gas": 25, "co2": 25},
)

world.add_unit(
    id="gas_plant",
    unit_type="power_plant",
    unit_operator_id="gas_operator",
    unit_params={
        "min_power": 0,
        "max_power": 4000,
        "efficiency": 0.6,
        "fuel_type": "natural gas",
        "emission_factor": 0.2,
        "additional_cost": 3.5,
        "technology": "gas",
        "bidding_strategies": {"EOM": "powerplant_energy_naive"},
    },
    forecaster=gas_forecaster,
)

# --- inelastic demand, also acting as buyer of the support contracts ---
demand_forecaster = DemandForecaster(index, demand=-demand_df["demand_EOM"])

world.add_unit(
    id="demand_unit",
    unit_type="demand",
    unit_operator_id="demand_operator",
    unit_params={
        # min_power is used as contract volume by the SupportStrategy
        "min_power": -4000,
        "max_power": -4000,
        "technology": "demand",
        "bidding_strategies": {"EOM": "demand_energy_naive", "Support": "support"},
        "bidding_params": {
            "contract_types": ["CFD", "PPA", "FIT", "market_premium"],
            "contract_amount_fraction": 1.0,
            "evaluation_frequency": rr.WEEKLY,
        },
    },
    forecaster=demand_forecaster,
)

### The five solar units

All solar units share the same capacity (1000 MW), the same availability time series from example_03
and therefore the same generation profile — only their revenue scheme differs. Their marginal cost is
~0 €/MWh, so on the EOM they all bid at price 0.

In [ ]:
solar_forecaster = PowerplantForecaster(
    index,
    availability=solar_availability,
    fuel_prices={},
)

solar_units = {
    "solar_cfd": ["CFD"],
    "solar_ppa": ["PPA"],
    "solar_fit": ["FIT"],
    "solar_mp": ["market_premium"],
    # the naive unit bids only on the EOM and takes the full price risk
    "solar_naive": [],
}

CONTRACT_VALUE = 80  # strike price of the support contracts in EUR/MWh

for unit_id, contract_types in solar_units.items():
    bidding_strategies = {"EOM": "powerplant_energy_naive"}
    bidding_params = {}
    if contract_types:
        bidding_strategies["Support"] = "support"
        bidding_params = {
            "contract_types": contract_types,
            "contract_value": CONTRACT_VALUE,
            "contract_amount_fraction": 1.0,
            "evaluation_frequency": rr.WEEKLY,
        }

    world.add_unit(
        id=unit_id,
        unit_type="power_plant",
        unit_operator_id="solar_operator",
        unit_params={
            "min_power": 0,
            "max_power": 1000,
            "efficiency": 1,
            "fuel_type": "renewable",
            "emission_factor": 0,
            "additional_cost": 0,
            "technology": "solar",
            "bidding_strategies": bidding_strategies,
            "bidding_params": bidding_params,
        },
        forecaster=solar_forecaster,
    )